In [2]:
import pandas as pd
import numpy as np
import pyxirr

# analysis window
START_DATE = pd.Timestamp("2023-12-29")
END_DATE   = pd.Timestamp("2025-12-31")

# input file paths - update these if files are stored elsewhere
sec_px_csv       = "sec_px.csv"
sec_metadata_csv = "sec_metadata.csv"
pm_txn_csvs      = [
    "pm1_transactions.csv",
    "pm2_transactions.csv",
    "pm3_transactions.csv",
]


## Part A - Top 3 & Bottom 3 securities by annualised TWR 

In [3]:
def load_sec_px(path: str) -> pd.DataFrame:
    px = pd.read_csv(
        path,
        parse_dates=["date"],
        dayfirst=False
    )
    px = px.set_index("date").sort_index()
    px = px.apply(pd.to_numeric, errors="coerce")
    return px
sec_px = load_sec_px(sec_px_csv)

# Part A uses an extended window to include Feb 2026 prices
PART_A_END_DATE = pd.Timestamp("2026-02-28")

def restrict_window(px: pd.DataFrame,
                    start: pd.Timestamp,
                    end: pd.Timestamp) -> pd.DataFrame:
    return px.loc[(px.index >= start) & (px.index <= end)]
sec_px_window = restrict_window(sec_px, START_DATE, PART_A_END_DATE)

def filter_eligible_securities(px: pd.DataFrame) -> pd.DataFrame:
    # Include only securities with a non-missing price at the last date of the
    # analysis window and at least one valid monthly return. This ensures we
    # evaluate currently investable securities — stocks delisted or exited
    # before the end of the window are excluded.
    end_prices = px.iloc[-1]
    px = px.loc[:, end_prices.notna()]

    returns = px.pct_change()
    has_returns = returns.notna().sum() > 0

    return px.loc[:, has_returns]
sec_px_clean = filter_eligible_securities(sec_px_window)

In [4]:
def load_sec_metadata(path: str) -> pd.DataFrame:
    meta = pd.read_csv(path)
    meta.columns = meta.columns.str.lower()
    meta["date"] = pd.to_datetime(meta["date"])
    return meta
sec_metadata = load_sec_metadata(sec_metadata_csv)
def build_name_map(meta: pd.DataFrame) -> pd.Series:
    latest = (
        meta.sort_values("date")
            .groupby("sedol")
            .tail(1)
    )
    return latest.set_index("sedol")["name"]

name_map = build_name_map(sec_metadata)


In [5]:
def compute_annualised_twr(px: pd.DataFrame) -> pd.DataFrame:
    """
    Compute annualised Time-Weighted Return (TWR) for each security
    using monthly price data.

    Parameters
    ----------
    px : pd.DataFrame
        Cleaned price panel with DatetimeIndex and SEDOL columns

    Returns
    -------
    pd.DataFrame
        Columns: sedol, annualised_twr, n_months
    """
    # Monthly returns
    returns = px.pct_change()

    results = []

    for sedol in returns.columns:
        r = returns[sedol].dropna()

        if len(r) == 0:
            continue

        # Total time-weighted return
        twr_total = (1 + r).prod() - 1

        # Annualise based on number of months observed
        annualised_twr = (1 + twr_total) ** (12 / len(r)) - 1

        results.append({
            "sedol": sedol,
            "annualised_twr": annualised_twr,
            "n_months": len(r)
        })

    return pd.DataFrame(results)

twr_df = compute_annualised_twr(sec_px_clean)


In [6]:
def attach_security_names(twr_df: pd.DataFrame,
                          name_map: pd.Series) -> pd.DataFrame:
    out = twr_df.copy()
    out["name"] = out["sedol"].map(name_map)
    return out
twr_df_named = attach_security_names(twr_df, name_map)

In [7]:
# Get final result
def get_top_bottom_securities(twr_named: pd.DataFrame,
                              n: int = 5) -> tuple[pd.DataFrame, pd.DataFrame]:
    ranked = twr_named.dropna(subset=["name"])

    top = (
        ranked.sort_values("annualised_twr", ascending=False)
              .head(n)
              .loc[:, ["name", "annualised_twr"]]
    )

    bottom = (
        ranked.sort_values("annualised_twr", ascending=True)
              .head(n)
              .loc[:, ["name", "annualised_twr"]]
    )

    return top, bottom

top_securities, bottom_securities = get_top_bottom_securities(twr_df_named)

def _fmt_pct(df: pd.DataFrame, col: str) -> pd.DataFrame:
    out = df.copy()
    out[col] = out[col].map('{:.2%}'.format)
    return out

print("Top 3 Securities:")
print(_fmt_pct(top_securities, 'annualised_twr').to_string(index=False))
print("\nBottom 3 Securities:")
print(_fmt_pct(bottom_securities, 'annualised_twr').to_string(index=False))


Top 3 Securities:
                                                name annualised_twr
            Zijin Gold International Company Limited        361.89%
                         Laopu Gold Co. Ltd. Class H        294.98%
Victory Giant Technology (HuiZhou) Co., Ltd. Class A        264.82%
                Pop Mart International Group Limited        202.50%
             Eoptolink Technology Inc., Ltd. Class A        191.92%

Bottom 3 Securities:
                                                  name annualised_twr
Chongqing Zhifei Biological Products Co., Ltd. Class A        -43.41%
                    Legend Biotech Corp. Sponsored ADR        -42.39%
                     iQIYI, Inc. Sponsored ADR Class A        -41.38%
                  Hygeia Healthcare Holdings Co., Ltd.        -37.59%
    Chongqing Taiji Industry (Group) Co., Ltd. Class A        -35.98%


### Part A Commentary

**Analysis window:** 2023-12-29 to 2026-02-28 | **Universe:** MSCI China eligible constituents with a valid price as of 2026-02-28 and at least one valid monthly return (currently investable securities only — delisted or index-exited names excluded)

**Top 3 performers** were led by gold names, reflecting the sustained rally in precious metals over the period. Zijin Gold International led with a **+361.89% annualised TWR**, followed by Laopu Gold (+294.98%) — both benefiting from strong gold prices and robust domestic demand for gold jewellery and investment products. Victory Giant Technology (+264.82%) was the sole technology name in the top 3, continuing to benefit from the broader AI hardware and semiconductor upcycle.

**Bottom 3 performers** remained concentrated in healthcare and media. Chongqing Zhifei Biological Products (**-43.41%**) and Legend Biotech (-42.39%) were weighed down by persistent regulatory and pricing headwinds from China's volume-based procurement programme. iQIYI (-41.38%) rounds out the bottom 3, reflecting continued pressure on China's online media sector amid weak consumer spending and intensifying competition.

**Return spread** between the best and worst performer was approximately **405 percentage points (annualised)**, reflecting very high cross-sectional dispersion within the MSCI China universe over this period.

## PART B - Cumulative Returns for each sector
Note: Sector returns reflect uncapped float-adjusted weights and may not align with official MSCI indices due to differences in weighting methodologies

Eg. Results exceed the official MSCI China Materials 10/50 Index (+132% over 2024–2025) as no constituent weight cap is applied, amplifying the contribution of top performers like Zijin Mining (+254%)

In [8]:
def compute_sector_cumulative_return(
    sec_px: pd.DataFrame,
    sec_metadata: pd.DataFrame,
    start_date: pd.Timestamp = None,
    end_date: pd.Timestamp = None,
    weight_lag_months: int = 1,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Compute chain-linked cumulative returns for each sector.

    Parameters
    ----------
    sec_px : pd.DataFrame
        Wide price panel with DatetimeIndex and SEDOL columns.
    sec_metadata : pd.DataFrame
        Must contain columns: sedol, sector, weight, date.
    start_date : pd.Timestamp, optional
        Inclusive start of the return window. Defaults to the full series.
    end_date : pd.Timestamp, optional
        Inclusive end of the return window. Defaults to the full series.
    weight_lag_months : int
        How many months to lag the metadata weights before merging with returns.
        Default is 1 (prior month's weights applied to current month's return).
    verbose : bool
        If True, prints diagnostic info such as merged row count.

    Returns
    -------
    pd.DataFrame
        Columns: sector, cumulative_return — sorted descending.
    """
    required_meta_cols = {'sedol', 'sector', 'weight', 'date'}
    missing = required_meta_cols - set(sec_metadata.columns)
    if missing:
        raise ValueError(f"sec_metadata is missing columns: {missing}")

    #  Step 1: Optionally restrict price window ─
    px = sec_px.copy()
    if start_date is not None:
        px = px.loc[px.index >= start_date]
    if end_date is not None:
        px = px.loc[px.index <= end_date]

    #  Step 2: Reshape sec_px wide → long ─
    px = px.reset_index()
    px_long = px.melt(id_vars='date', var_name='sedol', value_name='price')
    px_long = px_long.sort_values(['sedol', 'date'])

    #  Step 3: Compute monthly returns per stock 
    px_long['return'] = px_long.groupby('sedol')['price'].pct_change(fill_method=None)
    px_long = px_long.dropna(subset=['return'])

    #  Step 4: Convert both dates to year-month period 
    px_long['ym'] = pd.to_datetime(px_long['date']).dt.to_period('M')  

    # .dt purpose
    ## pd.to_datetime(px_long['date']) is a Series
    ## dt is the datetime accessor that exposes datetime operations for the Series


    meta = sec_metadata[['sedol', 'sector', 'weight', 'date']].copy()
    meta['ym'] = pd.to_datetime(meta['date']).dt.to_period('M')

    #  Step 5: Lag metadata weights 
    meta['ym'] = meta['ym'] + weight_lag_months

    #  Step 6: Merge on sedol + year-month ─
    df = px_long.merge(meta[['sedol', 'ym', 'sector', 'weight']],
                       on=['sedol', 'ym'], how='inner')

    if df.empty:
        raise ValueError(
            "Merge produced 0 rows. Check that sec_px and sec_metadata share "
            "overlapping SEDOLs and date ranges."
        )
    if verbose:
        print(f"Merged shape: {df.shape}")

    #  Step 7: Normalize weights within each sector-month ─
    df['w_norm'] = df.groupby(['sector', 'ym'])['weight'].transform(
        lambda x: x / x.sum()
    )

    #  Step 8: Compute weighted return per sector per month ─
    df['weighted_return'] = df['w_norm'] * df['return']
    monthly_sector = (
        df.groupby(['sector', 'ym'])['weighted_return']
        .sum()
        .reset_index()
    )

    #  Step 9: Chain link monthly returns per sector 
    monthly_sector['gross_return'] = 1 + monthly_sector['weighted_return']
    sector_cum = (
        monthly_sector.groupby('sector')['gross_return']
        .prod()
        .reset_index()
    )
    sector_cum['cumulative_return'] = sector_cum['gross_return'] - 1

    return sector_cum[['sector', 'cumulative_return']].sort_values(
        'cumulative_return', ascending=False
    )

sector_cumulative_returns = compute_sector_cumulative_return(
    sec_px, sec_metadata, verbose=True
)

display_df = sector_cumulative_returns.copy()
display_df['cumulative_return'] = display_df['cumulative_return'].map('{:.2%}'.format)
print("\nSector Cumulative Returns:")
print(display_df.to_string(index=False))


Merged shape: (15488, 7)

Sector Cumulative Returns:
                sector cumulative_return
             Materials           165.32%
            Financials            95.38%
Communication Services            82.25%
Information Technology            71.68%
                Energy            62.61%
           Industrials            48.51%
Consumer Discretionary            37.36%
           Health Care            25.35%
             Utilities            22.54%
           Real Estate             7.66%
      Consumer Staples            -3.08%


### Part B Commentary

**Analysis window:** 2023-12-29 to 2025-12-31 | **Weighting:** Float-adjusted, uncapped, lagged by 1 month

**Best performing sector: Materials (+165.32%)** is the standout sector, driven by strong commodity prices and exceptional single-stock contributions (e.g. Zijin Mining). The outsized return relative to other sectors partly reflects the uncapped weighting methodology, which amplifies contributions from large, high-performing constituents.

**Financials (+95.38%) and Communication Services (+82.25%)** ranked second and third, benefiting from improved sentiment around Chinese banks and a re-rating of internet/media names respectively.

**Consumer Staples was the only sector in negative territory (-3.08%)**, reflecting persistent weakness in consumer demand and deflationary pressures in staples categories across the period.

**Overall**, 10 of 11 sectors delivered positive cumulative returns over the window, suggesting broad-based market recovery, albeit with significant dispersion across sectors (Materials +165% vs. Consumer Staples -3%).

## Part C - IRR by PM

In [29]:
def get_price_on_date(sec_px: pd.DataFrame, sedol: str, date: pd.Timestamp) -> float:
    """Get closest available price on or before a given date."""
    if sedol not in sec_px.columns:
        return np.nan
    series = sec_px[sedol].dropna()
    available = series[series.index <= date]
    if available.empty:
        return np.nan
    return available.iloc[-1]

def compute_pm_irr(txn: pd.DataFrame, 
                   sec_px: pd.DataFrame, 
                   end_date: pd.Timestamp) -> float:
    """
    Compute XIRR for a single PM's transactions.

    Cash flows:
      - Buy  -> negative (cash out)
      - Sell -> positive (cash in)
      - Remaining holdings at end_date -> positive terminal value
    """
    records = []

    for _, row in txn.iterrows():
        price = get_price_on_date(sec_px, row['security_sedol'], row['txn_date'])
        if np.isnan(price):
            continue

        cash_flow = price * row['quantity']
        if row['side'].strip().lower() == 'buy':
            cash_flow *= -1  # cash out

        records.append({
            'date': row['txn_date'],
            'cash_flow': cash_flow
        })

    cf_df = pd.DataFrame(records).sort_values('date')

    # Terminal value: remaining holdings at end_date
    buys = txn[txn['side'].str.lower() == 'buy'].groupby('security_sedol')['quantity'].sum()
    sells = txn[txn['side'].str.lower() == 'sell'].groupby('security_sedol')['quantity'].sum()
    holdings = buys.subtract(sells, fill_value=0)
    holdings = holdings[holdings > 0]

    for sedol, qty in holdings.items():
        price = get_price_on_date(sec_px, sedol, end_date)
        if not np.isnan(price):
            cf_df = pd.concat([cf_df, pd.DataFrame([{
                'date': end_date,
                'cash_flow': price * qty  # positive: liquidate at end
            }])], ignore_index=True)

    cf_df = cf_df.sort_values('date').reset_index(drop=True)
    if cf_df.empty:
        return np.nan

    # XIRR requires at least one negative and one positive cash flow.
    flows = cf_df['cash_flow'].to_numpy()
    if not ((flows < 0).any() and (flows > 0).any()):
        return np.nan

    try:
        irr = pyxirr.xirr(cf_df['date'].tolist(), flows.tolist())
    except Exception:
        irr = np.nan

    return irr

results = {}
for i, path in enumerate(pm_txn_csvs, start=1):
    txn = pd.read_csv(path)
    txn.columns = txn.columns.str.lower()
    txn['txn_date'] = pd.to_datetime(txn['txn_date'], format="%Y-%m-%d")

    irr = compute_pm_irr(txn, sec_px, END_DATE)
    results[f'PM{i}'] = irr

print("PM IRR Results:")
for pm, irr in results.items():
    print(f"  {pm}: {irr:.2%}" if not np.isnan(irr) else f"  {pm}: N/A")
print('\n')

def stocks_held(txn: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    unique_sedol = txn['security_sedol'].unique()
    unique_stock_name = meta[meta['sedol'].isin(unique_sedol)][['sedol', 'sector', 'name']].drop_duplicates('sedol')
    print(f" {unique_stock_name} \n")

for i, path in enumerate(pm_txn_csvs, start = 1):
    txn = pd.read_csv(path)
    print("PM" f"{i} unique stocks: \n")
    # print(txn['security_sedol'].unique())
    stocks_held(txn, sec_metadata)
    



PM IRR Results:
  PM1: 47.43%
  PM2: 31.39%
  PM3: -7.66%


PM1 unique stocks: 

       sedol                  sector  \
8    B1H508               Materials   
109  BWVFT0              Financials   
301  BP3R35             Industrials   
509  BD5CK8  Information Technology   
628  BHQPRN  Consumer Discretionary   

                                                  name  
8            Zhaojin Mining Industry Co., Ltd. Class H  
109                Huatai Securities Co., Ltd. Class H  
301                   CRRC Corporation Limited Class A  
509  CETC Cyberspace Security Technology Co. Ltd. C...  
628       Offcn Education Technology Co., Ltd. Class A   

PM2 unique stocks: 

         sedol       sector                                               name
217    BP3R7B    Materials  Inner Mongolia Junzheng Energy & Chemical Grou...
292    BP3R3L   Financials        China Construction Bank Corporation Class A
558    BYW5MZ   Financials                 Bank of Hangzhou Co., Ltd. Class A
12132

### Part C Commentary

**Analysis window:** 2023-12-29 to 2025-12-31 | **Method:** XIRR on irregular cash flows; remaining holdings liquidated at 2025-12-31 prices

| PM | IRR |
|---|---|
| PM1 | +47.47% |
| PM2 | +31.41% |
| PM3 | -7.66% |

**PM1 (+47.47%)** delivered the strongest risk-adjusted return, suggesting a portfolio well-positioned towards the high-performing technology/materials names that led the market over the period.

**PM2 (+31.41%)** also generated a positive return but meaningfully below PM1, indicating either more conservative positioning, higher allocation to lagging sectors, or less favourable trade timing.

**PM3 (-7.66%)** was the only PM to record a negative IRR, indicating that the timing and composition of their transactions resulted in a net loss over the period, likely reflecting exposure to the underperforming healthcare or consumer staples names, or unfavourable entry/exit timing.

**IRR spread** between PM1 and PM3 is approximately **55 percentage points**, which is material and highlights the significant impact of stock selection and trade timing within the same broad MSCI China universe.

## Part D - Stock Returns by PM

In [31]:
def stock_returns(txn: pd.DataFrame, px: pd.DataFrame, end_date: pd.Timestamp = END_DATE) -> pd.DataFrame:
    results = []

    for sedol, group in txn.groupby('security_sedol'):
        if sedol not in px.columns:
            continue

        buys = group[group['side'].str.lower() == 'buy']
        sells = group[group['side'].str.lower() == 'sell']

        entry_date = buys['txn_date'].min()  # first buy

        total_bought = buys['quantity'].sum()
        total_sold = sells['quantity'].sum() if not sells.empty else 0
        still_held = total_bought > total_sold

        # If still held, use end_date as exit
        exit_date = end_date if still_held else sells['txn_date'].max()

        entry_price = px[sedol].asof(entry_date)
        exit_price = px[sedol].asof(exit_date)

        if pd.isna(entry_price) or pd.isna(exit_price):
            continue

        results.append({
            'sedol': sedol,
            'entry_date': entry_date,
            'exit_date': exit_date,
            'return': (exit_price - entry_price) / entry_price,
            'still_held': still_held
        })

    return pd.DataFrame(results)

for i, path in enumerate(pm_txn_csvs, start=1):
    txn = pd.read_csv(path)
    txn.columns = txn.columns.str.lower()
    txn['txn_date'] = pd.to_datetime(txn['txn_date'], format="%Y-%m-%d")

    returns_df = stock_returns(txn, sec_px)
    returns_df['return'] = returns_df['return'].map('{:.2%}'.format)
    print(f"PM{i} stock returns:")
    print(returns_df.to_string(index=False))
    print()


PM1 stock returns:
 sedol entry_date  exit_date  return  still_held
B1H508 2024-02-29 2025-12-31 264.94%        True
BD5CK8 2024-01-31 2025-12-31  24.01%        True
BHQPRN 2024-03-29 2025-12-31  -8.47%        True
BP3R35 2025-01-31 2025-12-31  -5.62%        True
BWVFT0 2023-12-29 2025-12-31 108.75%        True

PM2 stock returns:
 sedol entry_date  exit_date return  still_held
BNHPNT 2023-12-29 2025-02-28 32.87%       False
BP3R3L 2024-02-29 2025-12-31 47.00%        True
BP3R7B 2025-04-30 2025-12-31 -6.39%        True
BYW5MZ 2024-03-29 2025-12-31 50.88%        True

PM3 stock returns:
 sedol entry_date  exit_date return  still_held
BD5CNT 2024-03-29 2025-12-31 81.57%        True
BD5CPP 2024-07-31 2025-12-31 28.61%        True
BP3RBV 2024-05-31 2025-12-31 -5.55%        True
BQV3XP 2025-07-31 2025-12-31 70.35%        True
BR4VSK 2023-12-29 2025-10-31 50.24%       False

